## Finding all obliques in spatial cases from a database

The aim of this notebook is to find all obliques in spatial cases from the Estonian Reference corpus. This is needed to annotate them with semantic class using both rule based methods and LLMs. The code extracts data from Katrin Tsepelina's database [v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/001_verb_transactions/v33) with data extracted from the Estonian Reference corpus.

This code creates a new table in the database called 'spatial_obl' by extracting obliques in spatial cases (form + lemma + feats), their head verb (verb+compund) and sentence id.

The code also takes sentences from another database and adds them based on the sentence id to this one.

In [2]:
#imports
import sqlite3

### Create new table spatial_obl

In [2]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn = sqlite3.connect(filename)
cursor = conn.cursor()

In [6]:
#searches for obliques in spatial cases + head verb + sentence id
query = (f"CREATE TABLE spatial_obl AS " 
         f"SELECT transaction_row.id, transaction_row.head_id, transaction_row.form, lemma, transaction_row.feats, verb, verb_compound, sentence_id " 
         f"FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id WHERE transaction_row.deprel = 'obl' "
        f"AND (transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ?)")
cursor.execute(query, ('%adit%', '%ill%', '%in%', '%el%', '%all%', '%ad%', '%abl%'))

### Add sentences to spatial_obl table

In [3]:
# Connect to both databases
filename1 = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_sentences_20250220-130121.db"
filename2 = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

conn1 = sqlite3.connect(filename1)  # Source database
conn2 = sqlite3.connect(filename2)  # Target database

cursor1 = conn1.cursor()
cursor2 = conn2.cursor()

In [4]:
# Step 1: Retrieve sentences from database1
cursor1.execute("SELECT id, text FROM sentences")
sentences = cursor1.fetchall()  # List of (sentence_id, sentence)

In [6]:
# Step 2: add new column to database2
cursor2.execute("ALTER TABLE spatial_obl ADD COLUMN sentence TEXT")

In [9]:
# Step 1: Create a temporary table
cursor2.execute("CREATE TEMP TABLE temp_sentence (id INT PRIMARY KEY, sentence TEXT)")

# Step 2: Insert all values into the temp table
cursor2.executemany("INSERT INTO temp_sentence (id, sentence) VALUES (?, ?)", sentences)

# Step 3: Perform a fast join-based update
cursor2.execute("""
    UPDATE spatial_obl
    SET sentence = (SELECT sentence FROM temp_sentence WHERE temp_sentence.id = spatial_obl.sentence_id)
    WHERE EXISTS (SELECT 1 FROM temp_sentence WHERE temp_sentence.id = spatial_obl.sentence_id)
""")

conn2.commit()
conn1.close()
conn2.close()